<span style="font-weight:bold; font-size: 3rem; color:#333;">- Part 01: Feature Backfill for Train Delay Data (Two-Stage Model)</span>

## 🗒️ Overview

This notebook backfills historical features for the two-stage train delay model.

It performs the following steps:

1. **Choose train stations and time range**: define a list of LocationSignature codes for the Pendeltåg network and select a backfill window (e.g., last X days/months).
2. **Fetch historical TrainAnnouncement data** from Trafikverket's Open API using XML queries.
3. **Fetch auxiliary data** such as weather observations and ReasonCodes (if available) to enrich the dataset.
4. **Engineer features** such as delay minutes, time-of-day, day-of-week, recent delays, weather metrics, and reason categories.
5. **Save the resulting DataFrame** into a Hopsworks feature group for downstream training and inference.

> 🛠️ **Note**: You need to supply a valid Trafikverket API key. The API returns JSON when the request is sent in XML format. Replace placeholders where indicated.


### 📝 Imports

In [1]:
import os
import datetime
import pandas as pd
import requests
import hopsworks_utils
import datetime as dt
from dotenv import load_dotenv
#import hopsworks
from typing import Any, Dict, List, Optional, Tuple


# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

load_dotenv()


True

## 📡 Connect to Hopsworks Feature Store

In [2]:
# Optional: Hopsworks storage (not required)
try:
    project = hopsworks_utils.HopsworksInterface()
    print("Hopsworks login OK")
except Exception as e:
    project = None
    print("Hopsworks not configured / login failed (OK). Proceeding without it.")
    print("Reason:", repr(e))


HOPSWORKS_API_KEY exists: True
HOPSWORKS_API_KEY length: 81
2026-01-03 17:36:41,061 INFO: Initializing external client
2026-01-03 17:36:41,063 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-01-03 17:36:41,975 INFO: Python Engine initialized.

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/2182
Hopsworks login OK


## 🔑 Configure Trafikverket API and Helper Functions

In [3]:


from zoneinfo import ZoneInfo

# ============================================================
# Trafikverket Open API (v2) helpers
# ============================================================

TRAFIKVERKET_BASE_URL = os.getenv(
    "TRAFIKVERKET_BASE_URL",
    "https://api.trafikinfo.trafikverket.se/v2/data.json",
)
API_KEY_TRAFIK = os.getenv("API_KEY_TRAFIK")

# Define Stockholm timezone for explicit conversion
STOCKHOLM_TZ = ZoneInfo("Europe/Stockholm")

def _iso(ts: dt.datetime) -> str:
    """
    Format timestamp as Trafikverket ISO string without timezone.
    Converts to Stockholm local time first, then formats as naive.
    """
    if ts.tzinfo is not None:
        ts = ts.astimezone(STOCKHOLM_TZ)
    return ts.strftime("%Y-%m-%dT%H:%M:%S")


def build_request_xml(
    api_key: str,
    object_type: str,
    filter_xml: str,
    include_fields: List[str],
    limit: int = 10000,
    schema_version: str = "1", # Weather uses v1 often, but v2 is available.
    order_by: Optional[str] = None,
) -> str:
    # 1. Format INCLUDE tags
    include_xml = "".join(f"<INCLUDE>{f}</INCLUDE>" for f in include_fields)

    # 2. Add 'orderby' as an attribute to the QUERY tag, NOT as a child element
    orderby_attr = f'orderby="{order_by}"' if order_by else ""

    return f"""
<REQUEST>
  <LOGIN authenticationkey="{api_key}" />
  <QUERY objecttype="{object_type}" schemaversion="{schema_version}" limit="{limit}" {orderby_attr}>
    {filter_xml}
    {include_xml}
  </QUERY>
</REQUEST>
""".strip()


def query_trafikverket(xml_body: str, timeout: int = 60) -> Dict[str, Any]:
    headers = {"Content-Type": "text/xml; charset=utf-8"}
    r = requests.post(
        TRAFIKVERKET_BASE_URL,
        data=xml_body.encode("utf-8"),
        headers=headers,
        timeout=timeout,
    )
    if not r.ok:
        print(f"❌ API Error {r.status_code}:")
        print(r.text)
    r.raise_for_status()
    return r.json()


def _extract_result_list(resp: Dict[str, Any], object_type: str) -> List[Dict[str, Any]]:
    """Return list of objects from RESPONSE.RESULT[0][object_type]."""
    try:
        return resp["RESPONSE"]["RESULT"][0].get(object_type, [])
    except Exception:
        return []


def _chunk_time_windows(start: dt.datetime, end: dt.datetime, hours: int) -> List[Tuple[dt.datetime, dt.datetime]]:
    out = []
    cur = start
    while cur < end:
        nxt = min(end, cur + dt.timedelta(hours=hours))
        out.append((cur, nxt))
        cur = nxt
    return out


# ============================================================
# Fetch: TrainAnnouncement (ops: timetable + actual + est + cancel)
# ============================================================

TRAINANNOUNCE_FIELDS = [
    "ActivityId",
    "ActivityType",                 # Arrival / Departure
    "AdvertisedTrainIdent",         # train number
    "AdvertisedTimeAtLocation",     # scheduled
    "EstimatedTimeAtLocation",      # predicted
    "TimeAtLocation",               # actual (when available)
    "LocationSignature",            # station code
    "Canceled",
    "Deleted",
    "InformationOwner",
    "Deviation",                    # sometimes contains cause info
    "FromLocation",
    "ToLocation",
    "TrackAtLocation",
]


def fetch_train_announcements(
    station_codes: List[str],
    start_time: dt.datetime,
    end_time: dt.datetime,
    window_hours: int = 6,
    limit: int = 10000,
    api_key: str = API_KEY_TRAFIK,
) -> pd.DataFrame:
    """
    Backfill TrainAnnouncement data by splitting time into windows.
    """
    all_rows: List[Dict[str, Any]] = []
    windows = _chunk_time_windows(start_time, end_time, hours=window_hours)

    # Indentation for XML readability
    station_or = "\n      ".join([f'<EQ name="LocationSignature" value="{s}" />' for s in station_codes])

    for (ws, we) in windows:
        filter_xml = f"""
<FILTER>
  <AND>
    <GT name="AdvertisedTimeAtLocation" value="{_iso(ws)}" />
    <LT name="AdvertisedTimeAtLocation" value="{_iso(we)}" />
    <OR>
      {station_or}
    </OR>
  </AND>
</FILTER>
""".strip()

        xml = build_request_xml(
            api_key=api_key,
            object_type="TrainAnnouncement",
            filter_xml=filter_xml,
            include_fields=TRAINANNOUNCE_FIELDS,
            limit=limit,
            order_by="AdvertisedTimeAtLocation",
        )
        resp = query_trafikverket(xml)
        rows = _extract_result_list(resp, "TrainAnnouncement")
        all_rows.extend(rows)

    if not all_rows:
        return pd.DataFrame()

    df = pd.json_normalize(all_rows)

    # parse timestamps (some may be missing)
    for col in ["AdvertisedTimeAtLocation", "EstimatedTimeAtLocation", "TimeAtLocation"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    # standardize names
    df = df.rename(
        columns={
            "AdvertisedTimeAtLocation": "scheduled_time",
            "EstimatedTimeAtLocation": "estimated_time",
            "TimeAtLocation": "actual_time",
            "LocationSignature": "station_code",
            "AdvertisedTrainIdent": "train_id",
        }
    )

    # choose best available "observed" time for historical delay calc
    df["observed_time"] = df["actual_time"].fillna(df["estimated_time"])

    # delay in minutes
    df["delay_min"] = (df["observed_time"] - df["scheduled_time"]).dt.total_seconds() / 60.0

    # event_time = scheduled_time for point-in-time alignment
    df["event_time"] = df["scheduled_time"]

    # cancellation flag normalize
    df["is_canceled"] = df.get("Canceled", False).fillna(False).astype(bool)

    # basic calendar fields
    df["hour"] = df["event_time"].dt.hour
    df["dow"] = df["event_time"].dt.dayofweek
    df["date"] = df["event_time"].dt.date

    # keep only the core columns we need downstream
    keep = [
        "ActivityId",
        "ActivityType",
        "train_id",
        "OperationalTrainNumber",
        "event_time",
        "scheduled_time",
        "estimated_time",
        "actual_time",
        "observed_time",
        "station_code",
        "delay_min",
        "is_canceled",
        "Deleted",
        "InformationOwner",
        "Deviation",
        "FromLocation",
        "ToLocation",
        "TrackAtLocation",
        "hour",
        "dow",
        "date",
    ]
    keep = [c for c in keep if c in df.columns]
    df = df[keep].sort_values(["train_id", "event_time", "station_code"]).reset_index(drop=True)
    return df


# ============================================================
# Fetch: TrainMessage (incident-ish, optional)
# ============================================================

TRAINMESSAGE_FIELDS = [
    "ExternalDescription",
    "ReasonCodeText",
    "StartDateTime",
    "LastUpdateDateTime",
    "AffectedLocation",
    "EventId",
]

def fetch_train_messages(
    start_time: dt.datetime,
    end_time: dt.datetime,
    limit: int = 10000,
    api_key: str = API_KEY_TRAFIK,
) -> pd.DataFrame:
    """Fetch TrainMessage in a time window."""

    # Simple filter on StartDateTime only
    filter_xml = f"""
<FILTER>
  <AND>
    <LT name="StartDateTime" value="{_iso(end_time)}" />
    <GT name="StartDateTime" value="{_iso(start_time)}" />
  </AND>
</FILTER>
""".strip()

    xml = build_request_xml(
        api_key=api_key,
        object_type="TrainMessage",
        filter_xml=filter_xml,
        include_fields=TRAINMESSAGE_FIELDS,
        limit=limit,
        order_by="StartDateTime",
    )
    resp = query_trafikverket(xml)
    rows = _extract_result_list(resp, "TrainMessage")
    if not rows:
        return pd.DataFrame()

    df = pd.json_normalize(rows)

    # Parse dates
    for col in ["StartDateTime", "LastUpdateDateTime", "CreatedDateTime", "LastModifiedDateTime"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    return df


# ============================================================
# Weather: WeatherObservation (Limited retention ~7 days)
# ============================================================



# ============================================================
# Weather: WeatherObservation (Limited retention ~7 days)
# ============================================================

# CORRECTED FIELDS: 'MeasurementTime' is the correct field for time.
# We use Schema v2 notation (dot notation) for Temperature/Wind.
WEATHER_OBS_FIELDS = [
    "MeasurementTime",      # Fixed: was ObservationTime
    "ModifiedTime",
    "Id",
    "Air.Temperature",
    "Wind.Speed",
]

def fetch_weather_observations_last7d(
    start_time: dt.datetime,
    end_time: dt.datetime,
    limit: int = 10000,
    api_key: str = API_KEY_TRAFIK,
) -> pd.DataFrame:
    """
    WeatherObservation only keeps ~7 days historically.
    Returns empty if outside retention window.
    """
    filter_xml = f"""
<FILTER>
  <AND>
    <GT name="MeasurementTime" value="{_iso(start_time)}" />
    <LT name="MeasurementTime" value="{_iso(end_time)}" />
  </AND>
</FILTER>
""".strip()

    xml = build_request_xml(
        api_key=api_key,
        object_type="WeatherObservation",
        filter_xml=filter_xml,
        include_fields=WEATHER_OBS_FIELDS,
        limit=limit,
        order_by="MeasurementTime",
    )

    # We anticipate this might return empty for old dates, so we handle it gracefully.
    try:
        resp = query_trafikverket(xml)
        rows = _extract_result_list(resp, "WeatherObservation")
    except Exception as e:
        # If it fails (e.g. timeout or syntax), print but don't crash the whole pipeline
        print(f"⚠️ Weather fetch warning: {e}")
        return pd.DataFrame()

    if not rows:
        # Expected for 2023 dates (data purged after 7 days)
        return pd.DataFrame()

    df = pd.json_normalize(rows)
    for col in ["MeasurementTime", "ModifiedTime"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    df = df.rename(columns={"MeasurementTime": "weather_time"})
    return df

# ============================================================
# Join logic (ops ↔ incidents) + label creation
# ============================================================

def join_train_messages(
    ops_df: pd.DataFrame,
    msg_df: pd.DataFrame,
    time_buffer_min: int = 15,
) -> pd.DataFrame:
    """
    Best-effort incident join.
    Since TrainMessage has no EndTime in API, we assume it applies for 60 mins.
    """
    if ops_df.empty or msg_df.empty:
        ops_df["reason_code"] = None
        ops_df["reason_text"] = None
        return ops_df

    m = msg_df.copy()

    # Flatten AffectedLocation
    affected_col = None
    for cand in ["AffectedLocation", "AffectedLocation.LocationSignature"]:
        if cand in m.columns:
            affected_col = cand
            break

    if affected_col is None:
        ops_df["reason_code"] = None
        ops_df["reason_text"] = None
        return ops_df

    if affected_col == "AffectedLocation":
        m = m.explode("AffectedLocation")
        m["affected_station"] = m["AffectedLocation"].apply(
            lambda x: x.get("LocationSignature") if isinstance(x, dict) else None
        )
    else:
        m["affected_station"] = m[affected_col]

    m = m.dropna(subset=["affected_station"])

    # Create synthetic end time (Start + 1 hour)
    m["effective_end"] = m["StartDateTime"] + pd.Timedelta(hours=1)

    ops = ops_df.copy()
    ops["t_start"] = ops["event_time"] - pd.to_timedelta(time_buffer_min, unit="m")
    ops["t_end"] = ops["event_time"] + pd.to_timedelta(time_buffer_min, unit="m")

    # Select columns to merge
    cols_to_use = ["StartDateTime", "effective_end", "affected_station", "ReasonCodeText", "ExternalDescription"]
    if "ReasonCode" in m.columns:
        cols_to_use.append("ReasonCode")

    merged = ops.merge(
        m[cols_to_use],
        left_on="station_code",
        right_on="affected_station",
        how="left",
    )

    # Update overlap logic
    overlap = (
        (merged["StartDateTime"].isna())
        | ((merged["StartDateTime"] <= merged["t_end"]) & (merged["effective_end"] >= merged["t_start"]))
    )
    merged = merged[overlap].copy()

    # Deduplicate
    merged["msg_rank_time"] = merged["StartDateTime"].fillna(pd.Timestamp.min)
    merged = merged.sort_values(["train_id", "event_time", "msg_rank_time"], ascending=[True, True, False])
    merged = merged.drop_duplicates(subset=["train_id", "event_time", "station_code"], keep="first")

    # Clean up
    merged = merged.rename(columns={"ReasonCodeText": "reason_text", "ExternalDescription": "reason_desc"})
    drop_cols = ["t_start", "t_end", "affected_station", "msg_rank_time", "effective_end"]
    merged = merged.drop(columns=[c for c in drop_cols if c in merged.columns])

    return merged


def add_labels(
    df: pd.DataFrame,
    horizon_min: int = 60,
    delay_threshold_min: int = 10,
) -> pd.DataFrame:
    """
    Create:
    - predictive label: will delay exceed threshold within next horizon?
    - reactive labels: final_delay for that train/day, additional_delay from now
    """
    if df.empty:
        return df

    out = df.copy()
    out["train_run_id"] = out["train_id"].astype(str) + "_" + out["date"].astype(str)

    # final delay per train run: use last known delay
    out["final_delay_min"] = out.groupby("train_run_id")["delay_min"].transform("last")
    out["additional_delay_min"] = out["final_delay_min"] - out["delay_min"]

    out = out.sort_values(["train_run_id", "event_time"]).reset_index(drop=True)
    horizon = pd.Timedelta(minutes=horizon_min)

    pred_flags = []
    for _, g in out.groupby("train_run_id", sort=False):
        times = g["event_time"].to_numpy()
        delays = g["delay_min"].to_numpy()
        y = []
        for i in range(len(g)):
            t0 = times[i]
            j = i
            max_d = -1e9
            while j < len(g) and (times[j] - t0) <= horizon:
                if pd.notna(delays[j]):
                    if delays[j] > max_d:
                        max_d = delays[j]
                j += 1
            y.append(1 if max_d >= delay_threshold_min else 0)
        pred_flags.extend(y)

    out["y_delay_within_horizon"] = pred_flags
    return out

## 🕒 Define Backfill Parameters

In [4]:
#station_codes = ["Cst", "Sci", "Mr", "U", "Sod", "Tål"]
# Corrected station codes for Stockholm Commuter Rail (Pendeltåg)
station_codes = [
    "Cst",  # Stockholm C
    "Sci",  # Stockholm City
    "Sst",  # Stockholm Södra (Fixed from Sod)
    "Åbe",  # Årstaberg
    "Äs",   # Älvsjö
    "Tul",  # Tullinge (Fixed from Tål)
    # Add others as needed: 'Tu' (Tumba), 'Söc' (Södertälje C)
]

# NOTE: Trafikverket timestamps are local-ish (no tz). We use UTC here for convenience.
# FIX: Ensure end_time is not in the future, as the Colab environment's clock might be advanced.
# Setting a specific recent date to avoid issues with advanced system clocks.
# Use current time (or a date within the last ~12 months)
end_time = dt.datetime.now(dt.timezone.utc)
start_time = end_time - dt.timedelta(days=7)  # Try 7 days first to test

print(f"Backfilling {start_time} -> {end_time}")

# Label settings
HORIZON_MIN = 60
DELAY_THRESHOLD_MIN = 10

print(f"Backfilling {start_time} \u2192 {end_time} for {len(station_codes)} stations")
print(f"Labels: horizon={HORIZON_MIN} min, threshold={DELAY_THRESHOLD_MIN} min")

Backfilling 2025-12-27 16:36:52.138363+00:00 -> 2026-01-03 16:36:52.138363+00:00
Backfilling 2025-12-27 16:36:52.138363+00:00 → 2026-01-03 16:36:52.138363+00:00 for 6 stations
Labels: horizon=60 min, threshold=10 min


In [5]:
ops_df = fetch_train_announcements(
    station_codes=station_codes,
    start_time=start_time,
    end_time=end_time,
    window_hours=6,
    limit=10000,
)

print("ops rows:", len(ops_df))
display(ops_df.head())

# Incidents (optional)
msg_df = fetch_train_messages(start_time=start_time, end_time=end_time)
print("messages rows:", len(msg_df))

ops_df = join_train_messages(ops_df, msg_df)

# Weather (optional; only last ~7 days coverage)
weather_df = fetch_weather_observations_last7d(
    start_time=max(start_time, end_time - dt.timedelta(days=7)),
    end_time=end_time,
)
print("weather rows:", len(weather_df))

# NOTE: We do not hard-join weather here yet because weather datasets vary.
# We keep weather_df as a separate artifact for now.

# Labels
df = add_labels(ops_df, horizon_min=HORIZON_MIN, delay_threshold_min=DELAY_THRESHOLD_MIN)

print("labeled rows:", len(df))
display(df.head())


ops rows: 19419


,ActivityId,ActivityType,train_id,event_time,scheduled_time,estimated_time,actual_time,observed_time,station_code,delay_min,is_canceled,Deleted,InformationOwner,Deviation,FromLocation,ToLocation,TrackAtLocation,hour,dow,date
0,1500adde-075d-66fb-08de-3ba1e6e40499,Avgang,10,2025-12-30 12:11:00+01:00,2025-12-30 12:11:00+01:00,NaT,2025-12-30 12:10:00+01:00,2025-12-30 12:10:00+01:00,Cst,-1.0,False,False,SJ,[Spårändrat],[Cst],"[U, Gä, Suc, Ös, Du]",6,12,1,2025-12-30
1,1500adde-075d-66fb-08de-3c62905b765c,Avgang,10,2025-12-31 12:11:00+01:00,2025-12-31 12:11:00+01:00,NaT,2025-12-31 12:13:00+01:00,2025-12-31 12:13:00+01:00,Cst,2.0,False,False,SJ,[Spårändrat],[Cst],"[U, Gä, Suc, Ös, Du]",11,12,2,2025-12-31
2,1500adde-075d-66fb-08de-3d1a9514e8b3,Avgang,10,2026-01-01 12:11:00+01:00,2026-01-01 12:11:00+01:00,NaT,NaT,NaT,Cst,NaN,True,False,SJ,[Inställt],[Cst],"[U, Gä, Suc, Ös, Du]",x,12,3,2026-01-01
3,1500adde-075d-66fb-08de-3de271d5a7e9,Avgang,10,2026-01-02 12:11:00+01:00,2026-01-02 12:11:00+01:00,NaT,NaT,NaT,Cst,NaN,True,False,SJ,"[Inställt, Oväder]",[Cst],"[U, Gä, Suc, Ös, Du]",x,12,4,2026-01-02
4,1500adde-075d-66fb-08de-3eac570d0035,Avgang,10,2026-01-03 12:11:00+01:00,2026-01-03 12:11:00+01:00,NaT,NaT,NaT,Cst,NaN,True,False,SJ,[Inställt],[Cst],"[U, Gä, Suc, Ös, Du]",x,12,5,2026-01-03


messages rows: 0
❌ API Error 400:
{ "RESPONSE":{"RESULT":[{ "ERROR":{"SOURCE":"Request","MESSAGE":"Invalid query attribute WeatherObservation.MeasurementTime"}}]}}
⚠️ Weather fetch warning: 400 Client Error: Bad Request for url: https://api.trafikinfo.trafikverket.se/v2/data.json
weather rows: 0
labeled rows: 19419


,ActivityId,ActivityType,train_id,event_time,scheduled_time,estimated_time,actual_time,observed_time,station_code,delay_min,...,TrackAtLocation,hour,dow,date,reason_code,reason_text,train_run_id,final_delay_min,additional_delay_min,y_delay_within_horizon
0,1500adde-075d-66fb-08de-3c62916f97eb,Avgang,10209,2025-12-31 17:22:00+01:00,2025-12-31 17:22:00+01:00,NaT,2025-12-31 17:22:00+01:00,2025-12-31 17:22:00+01:00,Cst,0.0,...,15a,17,2,2025-12-31,None,None,10209_2025-12-31,0.0,0.0,0
1,1500adde-075d-66fb-08de-3d1a95eff33e,Avgang,10209,2026-01-01 17:18:00+01:00,2026-01-01 17:18:00+01:00,NaT,2026-01-01 17:18:00+01:00,2026-01-01 17:18:00+01:00,Cst,0.0,...,15a,17,3,2026-01-01,None,None,10209_2026-01-01,0.0,0.0,0
2,1500adde-075d-66fb-08de-3eac57f45a52,Avgang,10209,2026-01-03 17:22:00+01:00,2026-01-03 17:22:00+01:00,NaT,2026-01-03 17:22:00+01:00,2026-01-03 17:22:00+01:00,Cst,0.0,...,15a,17,5,2026-01-03,None,None,10209_2026-01-03,0.0,0.0,0
3,1500adde-075d-66fb-08de-3d1a962c227f,Avgang,10247,2026-01-01 19:23:00+01:00,2026-01-01 19:23:00+01:00,NaT,2026-01-01 19:33:00+01:00,2026-01-01 19:33:00+01:00,Cst,10.0,...,15a,19,3,2026-01-01,None,None,10247_2026-01-01,10.0,0.0,1
4,1500adde-075d-66fb-08de-3c6291b5461a,Avgang,10249,2025-12-31 20:23:00+01:00,2025-12-31 20:23:00+01:00,NaT,2025-12-31 20:23:00+01:00,2025-12-31 20:23:00+01:00,Cst,0.0,...,14a,20,2,2025-12-31,None,None,10249_2025-12-31,0.0,0.0,0


In [ ]:
#Fixing some values in the dataframe that hopsworks does not like
print(weather_df.info())
#print(df.info())
df.fillna({
    "reason_code": 0,
    "reason_text": ""
}, inplace=True)






<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Empty DataFrame
None
<class 'pandas.core.series.Series'>
RangeIndex: 19349 entries, 0 to 19348
Series name: reason_text
Non-Null Count  Dtype 
--------------  ----- 
19349 non-null  object
dtypes: object(1)
memory usage: 151.3+ KB
None


## 🧬 Create Feature Group and Insert Historical Data

In [ ]:
import os

# Canonical output (truth + labels)
out_path = "data/train_stop_events_labeled.parquet"
out_path_weather = "data/weather_features.parquet"
print(os.path.dirname(out_path))

# Create the directory if it doesn't exist
os.makedirs(os.path.dirname(out_path), exist_ok=True)

if df is None or df.empty:
    print("No labeled data produced; nothing to save.")
else:
    df.to_parquet(out_path, index=False)
    weather_df.to_parquet(out_path_weather, index=False)
    print("✅ Saved canonical labeled table:", out_path)

# Optional: store in Hopsworks as well (still OK as storage; no training here)
if project is not None and df is not None and not df.empty:
    project.push(df, "train_stop_events_labeled", ["ActivityId"], "Canonical ops dataset + labels (no model training features yet)")
    print("Train events stored at hopsworks")
    


if project is not None and df is not None and not weather_df.empty:
    raise Exception("TODO: make sure to assign primary key and check weather_df formatting before removing this line and uploading weather features to hopsworks")
    project.push(weather_df, "weather_features", ["PRIMARYKEY"], "Historical Weather data(no model training features yet)")
    

data
✅ Saved canonical labeled table: data/train_stop_events_labeled.parquet
Feature Group created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai:443/p/2182/fs/2134/fg/2238
2026-01-03 17:31:12,245 INFO: Computing insert statistics
Inserted historical data into feature group "train_delay_features"
Train events stored at hopsworks
